# 🍏 Basic Retrieval-Augmented Generation (RAG) with AIProjectClient 🍎

In this notebook, we'll demonstrate a **basic RAG** flow using:
- **`azure-ai-projects`** (AIProjectClient)
- **`azure-ai-inference`** (Embeddings, ChatCompletions)
- **`azure-ai-search`** (for vector or hybrid search)

Our theme is **Health & Fitness** 🍏 so we’ll create a simple set of health tips, embed them, store them in a search index, then do a query that retrieves relevant tips, and pass them to an LLM to produce a final answer.

> **Disclaimer**: This is not medical advice. For real health questions, consult a professional.

## What is RAG?
[Retrieval-Augmented Generation](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/retrieval-augmented-generation#what-is-rag) (RAG) is a technique where the LLM (Large Language Model) uses relevant retrieved text chunks from your data to craft a final answer. This helps ground the model's response in real data, reducing hallucinations.


<img src="./seq-diagrams/3-basic-rag.png" width="75%"/>

## 1. Setup
We'll import libraries, load environment variables, and create an `AIProjectClient`.

> #### Complete [2-embeddings.ipynb](2-embeddings.ipynb) notebook before starting this one


In [ ]:
import os
from dotenv import load_dotenv

# azure-ai-projects
from azure.ai.projects import AIProjectClient
from azure.identity import InteractiveBrowserCredential

# For chat and system messages
from azure.ai.inference.models import UserMessage, SystemMessage

# For vector search or hybrid search
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential
from pathlib import Path

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir / '.env')

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
chat_model = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")
embedding_model = os.environ.get("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-3-small")
search_index_name = os.environ.get("SEARCH_INDEX_NAME", "healthtips-index")
tenant_id = os.environ.get("TENANT_ID")

print(f"🔑 Using Tenant ID: {tenant_id}")

try:
    print("🌐 Using browser-based authentication to bypass Azure CLI cache issues...")
    
    # Use only InteractiveBrowserCredential with the specific tenant
    credential = InteractiveBrowserCredential(tenant_id=tenant_id)
    
    # Create the project client using endpoint
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )
    print("✅ AIProjectClient created successfully!")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)
    print("💡 Please complete the browser authentication prompt that should appear")

## 2. Create Sample Health Data
We'll create a few short doc chunks. In a real scenario, you might read from CSV or PDFs, [chunk them up](https://learn.microsoft.com/en-us/azure/search/vector-search-how-to-chunk-documents), embed them, and store them in your search index.


In [ ]:
health_tips = [
    {
        "id": "doc1",
        "content": "Daily 30-minute walks help maintain a healthy weight and reduce stress.",
        "source": "General Fitness"
    },
    {
        "id": "doc2",
        "content": "Stay hydrated by drinking 8-10 cups of water per day.",
        "source": "General Fitness"
    },
    {
        "id": "doc3",
        "content": "Consistent sleep patterns (7-9 hours) improve muscle recovery.",
        "source": "General Fitness"
    },
    {
        "id": "doc4",
        "content": "For cardio endurance, try interval training like HIIT.",
        "source": "Workout Advice"
    },
    {
        "id": "doc5",
        "content": "Warm up with dynamic stretches before running to reduce injury risk.",
        "source": "Workout Advice"
    },
    {
        "id": "doc6",
        "content": "Balanced diets typically include protein, whole grains, fruits, vegetables, and healthy fats.",
        "source": "Nutrition"
    },
]
print("Created a small list of health tips.")

## 3.0. Create or Reset the Index
When creating a vector field in Azure AI Search, the **field definition** must include a `vector_search_profile` property that points to a matching profile name in your vector search settings.

We'll define a helper function to create (or reset) a vector index with an [HNSW algorithm](https://learn.microsoft.com/en-us/azure/search/vector-search-ranking#algorithms-used-in-vector-search) config.


In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

def create_healthtips_index(
        endpoint: str, api_key: str, index_name: str, 
        dimension: int = 1536 # if using text-embedding-3-small
        ):
    """Create or update a search index for health tips with vector search capability."""
    
    index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
    
    # Try to delete existing index
    try:
        index_client.delete_index(index_name)
        print(f"Deleted existing index: {index_name}")
    except Exception:
        pass  # Index doesn't exist yet
        
    # Define vector search configuration
    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(
                    m=4,
                    ef_construction=400,
                    ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE
                )
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="myHnswProfile",
                algorithm_configuration_name="myHnsw"
            )
        ]
    )
    
    # Define fields
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String),
        SearchField(
            name="embedding", 
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=dimension,
            vector_search_profile_name="myHnswProfile" 
        ),
    ]
    
    # Create index definition
    index_def = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search
    )
    
    # Create the index
    index_client.create_index(index_def)
    print(f"✅ Created or reset index: {index_name}")

## 3.1. Setup Search Connection & Upload Health Tips 🏋️

Now we'll:
1. **Get search connection** from AI Foundry project (or use environment variables)
2. **Create the search index** with vector search capability
3. **Generate embeddings** for each health tip
4. **Upload** the tips with their embeddings to the index


In [ ]:
from azure.ai.projects.models import ConnectionType

# Step 1: Try to get search connection from project
# First, let's see what connections are available
print("🔍 Checking for Azure AI Search connections in project...")
search_conn = None

try:
    # Try to get default Azure AI Search connection
    search_conn = project_client.connections.get_default(
        connection_type=ConnectionType.AZURE_AI_SEARCH, 
        include_credentials=True
    )
    print("✅ Found default Azure AI Search connection in project")
except ValueError as e:
    print(f"ℹ️ No default Azure AI Search connection found: {e}")
    print("ℹ️ Checking for any Azure AI Search connections...")
    
    # List all Azure AI Search connections
    search_connections = list(project_client.connections.list(
        connection_type=ConnectionType.AZURE_AI_SEARCH
    ))
    
    if search_connections:
        # Use the first available connection
        conn_name = search_connections[0].name
        print(f"✅ Found Azure AI Search connection: {conn_name}")
        search_conn = project_client.connections.get(
            conn_name, 
            include_credentials=True
        )
    else:
        print("⚠️ No Azure AI Search connections found in project")
        print("💡 Will use environment variables (AZURE_AI_SEARCH_ENDPOINT and AZURE_AI_SEARCH_API_KEY)")

# Extract endpoint and key
if search_conn:
    # Got connection from project
    if hasattr(search_conn, 'endpoint_url'):
        search_endpoint = search_conn.endpoint_url
    elif hasattr(search_conn, 'target'):
        search_endpoint = search_conn.target
    else:
        raise RuntimeError("❌ Cannot find endpoint in search connection")

    # Try multiple ways to get the API key
    search_key = None
    if hasattr(search_conn, 'key'):
        search_key = search_conn.key
    elif hasattr(search_conn, 'credentials') and hasattr(search_conn.credentials, 'key'):
        search_key = search_conn.credentials.key
    elif hasattr(search_conn, 'credentials') and hasattr(search_conn.credentials, 'api_key'):
        search_key = search_conn.credentials.api_key
    
    if not search_key:
        raise RuntimeError("❌ Cannot find API key in search connection")
    
    print(f"✅ Using search endpoint from project: {search_endpoint}")
else:
    # Fall back to environment variables
    search_endpoint = os.environ.get("AZURE_AI_SEARCH_ENDPOINT")
    search_key = os.environ.get("AZURE_AI_SEARCH_API_KEY")
    
    if not search_endpoint or not search_key:
        raise RuntimeError(
            "❌ No Azure AI Search connection found in project and "
            "environment variables AZURE_AI_SEARCH_ENDPOINT or AZURE_AI_SEARCH_API_KEY are not set"
        )
    
    print(f"✅ Using search endpoint from environment: {search_endpoint}")

# Step 2: Get embeddings client and check embedding length
with project_client.get_openai_client(api_version="2024-10-21") as embeddings_client:
    sample_doc = health_tips[0]
    emb_response = embeddings_client.embeddings.create(
        model=embedding_model,
        input=[sample_doc["content"]]
    )
    embedding_length = len(emb_response.data[0].embedding)
    print(f"✅ Got embedding length: {embedding_length}")

# Step 3: Create the index
create_healthtips_index(
    endpoint=search_endpoint,
    api_key=search_key,
    index_name=search_index_name,
    dimension=embedding_length
)

# Step 4: Create search client for uploading documents
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_key)
)
print("✅ Created search client")

# Step 5: Embed and upload documents
search_docs = []
with project_client.get_openai_client(api_version="2024-10-21") as embed_client:
    for doc in health_tips:
        # Get embedding for document content
        emb_response = embed_client.embeddings.create(
            model=embedding_model,
            input=[doc["content"]]
        )
        emb_vec = emb_response.data[0].embedding
        
        # Create document with embedding
        search_docs.append({
            "id": doc["id"],
            "content": doc["content"],
            "source": doc["source"],
            "embedding": emb_vec,
        })

# Upload documents to index
result = search_client.upload_documents(documents=search_docs)
print(f"✅ Uploaded {len(search_docs)} documents to search index '{search_index_name}'")


## 4. Basic RAG Flow
Now we'll implement the core [RAG pattern](https://learn.microsoft.com/en-us/azure/search/retrieval-augmented-generation-overview): **Retrieve** relevant documents, then **Generate** an answer using those documents as context.

### 4.1. Retrieve
When a user queries, we:
1. **Embed** the user's question using the same embedding model
2. **Search** the vector index to find semantically similar documents
3. **Return** the top-k most relevant documents

### 4.2. Generate Answer
We then pass the retrieved documents as context to the chat model, which:

1. **Grounds** its response in the provided facts

2. **Answers** only based on the retrieved information> 💡 In production scenarios, you'd use [hybrid search](https://learn.microsoft.com/en-us/azure/search/hybrid-search-overview), advanced chunking, and [semantic ranking](https://learn.microsoft.com/en-us/azure/search/semantic-search-overview). This example keeps it simple for learning.

3. **Avoids hallucination** by staying within the given context

In [ ]:
from azure.search.documents.models import VectorizedQuery

def rag_chat(query: str, top_k: int = 3) -> str:
    """
    Perform RAG: Retrieve relevant documents and generate an answer.
    
    Args:
        query: User's question
        top_k: Number of documents to retrieve (default: 3)
    
    Returns:
        Generated answer grounded in retrieved documents
    """
    # 1) Embed user query
    with project_client.get_openai_client(api_version="2024-10-21") as embed_client:
        user_vec_response = embed_client.embeddings.create(
            model=embedding_model,
            input=[query]
        )
        user_vec = user_vec_response.data[0].embedding

    # 2) Vector search using VectorizedQuery
    vector_query = VectorizedQuery(
        vector=user_vec,
        k_nearest_neighbors=top_k,
        fields="embedding"
    )

    results = search_client.search(
        search_text="",  # Optional text query
        vector_queries=[vector_query],
        select=["content", "source"]  # Only retrieve fields we need
    )

    # gather the top docs
    top_docs_content = []
    for r in results:
        c = r["content"]
        s = r["source"]
        top_docs_content.append(f"Source: {s} => {c}")

    # 3) Chat with retrieved docs
    system_text = (
        "You are a health & fitness assistant.\n"
        "Answer user questions using ONLY the text from these docs.\n"
        "Docs:\n"
        + "\n".join(top_docs_content)
        + "\nIf unsure, say 'I'm not sure'.\n"
    )

    with project_client.get_openai_client(api_version="2024-10-21") as chat_client:
        response = chat_client.chat.completions.create(
            model=chat_model,
            messages=[
                SystemMessage(content=system_text),
                UserMessage(content=query)
            ]
        )
    return response.choices[0].message.content

## 5. Try a Query 🎉
Let's do a question about cardio for busy people.


In [ ]:
user_query = "What's a good short cardio routine for me if I'm busy?"
#user_query = "What's a good balanced diet plan for someone with a hectic schedule?"
answer = rag_chat(user_query)
print("🗣️ User Query:", user_query)
print("🤖 RAG Answer:", answer)

## 6. Conclusion
We've demonstrated a **basic RAG** pipeline with:
- **Embedding** docs & storing them in **Azure AI Search**.
- **Retrieving** top docs for user question.
- **Chat** with the retrieved docs.

🔎 You can expand this by adding advanced chunking, more robust retrieval, and quality checks. Enjoy your healthy coding! 🍎